In [32]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import joblib

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df_air = pd.read_excel('/content/IndiaDataset.xlsx')
df_air = df_air[['Ozone', 'SO2', 'NO2', 'PM10', 'PM2.5']]

In [ ]:
df_air.head()

In [ ]:
scaler_lstm = MinMaxScaler()
scaled_air = scaler_lstm.fit_transform(df_air)
joblib.dump(scaler_lstm, "scaler_lstm.pkl")

In [ ]:
data['From Date'] = pd.to_datetime(data['From Date'], format='%d-%m-%Y %H:%M') # Specify the correct format for day first
numeric_features = data.select_dtypes(include=np.number).columns

scaler_lstm = MinMaxScaler()
# Only select numeric features for scaling
scaled_data = scaler_lstm.fit_transform(data[numeric_features])
joblib.dump(scaler_lstm, f"/content/drive/MyDrive/PBL2_files/scaler_lstm.pkl")

In [ ]:
look_back = 24
X, y = [], []

for i in range(look_back, len(scaled_data)):
    X.append(scaled_data[i - look_back:i, :])
    y.append(scaled_data[i, :])  # Predict next step

X, y = np.array(X), np.array(y)

In [ ]:
model1 = Sequential([
    LSTM(50, return_sequences=True, input_shape=(look_back, X.shape[2])),
    Dropout(0.2),
    LSTM(50),
    Dropout(0.2),
    Dense(X.shape[2])  # Predict all pollutants
])

model1.compile(optimizer='adam', loss='mse')
model1.fit(X, y, epochs=50, batch_size=32, validation_split=0.2)

In [ ]:
model1.save(f"/content/drive/MyDrive/PBL2_files/trained_lstm_model.h5")
print("✅ LSTM Model Trained & Saved Successfully!")

In [ ]:
df_health = pd.read_csv('/content/air_quality_health_impact_data.csv')

In [ ]:
df_health.head()

In [ ]:
features = ['PM2_5', 'PM10', 'NO2', 'SO2', 'O3']
X_health = df_health[features]
y_health = df_health['HealthImpactScore']

In [ ]:
scaler_rf = StandardScaler()
X_health_scaled = scaler_rf.fit_transform(X_health)
joblib.dump(scaler_rf, f"/content/drive/MyDrive/PBL2_files/scaler_rf.pkl")

In [ ]:
model2 = RandomForestRegressor(n_estimators=100, random_state=42)
model2.fit(X_health_scaled, y_health)

In [ ]:
joblib.dump(model2, f"/content/drive/MyDrive/PBL2_files/trained_health_score_rf_model.pkl")
print("✅ Random Forest Model Trained & Saved Successfully!")